In [ ]:


"""
================================================================================
Q4-Q9: Complete DQN and DDQN Implementation for HalfCheetah
================================================================================

This file contains comprehensive implementations for:
- Q4: Early Learning Reward Decomposition
- Q5: Instability Identification in Value Estimates
- Q6: Targeted Algorithmic Modifications
- Q7: Confidence-Driven Reduction in Exploration
- Q8: Performance Visualization and Comparison
- Q9: Summary and Analysis

Both DQN and DDQN implementations are included with full tracking and analysis.
================================================================================
"""

import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import pickle
from collections import deque, defaultdict
import random

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

# Set seeds
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")


# ============================================================================
# NETWORK ARCHITECTURE
# ============================================================================

class DQNNetwork(nn.Module):
    """Deep Q-Network for HalfCheetah"""

    def __init__(self, state_dim, action_dim, hidden_dims=[256, 256, 128]):
        super(DQNNetwork, self).__init__()

        layers = []
        input_dim = state_dim

        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(input_dim, hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.LayerNorm(hidden_dim))
            input_dim = hidden_dim

        layers.append(nn.Linear(input_dim, action_dim))
        self.network = nn.Sequential(*layers)

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.orthogonal_(module.weight, gain=np.sqrt(2))
            nn.init.constant_(module.bias, 0.0)

    def forward(self, state):
        return self.network(state)


# ============================================================================
# EXPERIENCE REPLAY
# ============================================================================

class ExperienceReplayBuffer:
    """Experience Replay Buffer for DQN"""

    def __init__(self, capacity=100000):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)

        return (
            np.array(states, dtype=np.float32),
            np.array(actions, dtype=np.int64),
            np.array(rewards, dtype=np.float32),
            np.array(next_states, dtype=np.float32),
            np.array(dones, dtype=np.float32)
        )

    def __len__(self):
        return len(self.buffer)


# ============================================================================
# ACTION DISCRETIZER (reuse from Q1)
# ============================================================================

class ActionDiscretizer:
    """Discretizes continuous action space"""

    def __init__(self, action_dim, action_low, action_high, num_bins_per_dim=3):
        self.action_dim = action_dim
        self.num_bins_per_dim = num_bins_per_dim
        self.discrete_values = np.linspace(action_low, action_high, num_bins_per_dim)
        self.num_discrete_actions = num_bins_per_dim ** action_dim

        grids = np.meshgrid(*[self.discrete_values for _ in range(self.action_dim)])
        self.discrete_actions = np.stack([grid.flatten() for grid in grids], axis=1)

        print(f"Action Discretization: {self.num_discrete_actions} discrete actions")

    def discretize_action(self, action_index):
        return self.discrete_actions[action_index]


# ============================================================================
# DQN AGENT
# ============================================================================

class DQNAgent:
    """DQN Agent with comprehensive tracking for Q4-Q7"""

    def __init__(self, state_dim, action_dim,
                 learning_rate=1e-4,
                 gamma=0.99,
                 epsilon_start=1.0,
                 epsilon_end=0.01,
                 epsilon_decay=0.995,
                 buffer_capacity=100000,
                 batch_size=64,
                 target_update_freq=100,
                 use_double_dqn=False):

        self.state_dim = state_dim
        self.action_dim = action_dim
        self.gamma = gamma
        self.epsilon = epsilon_start
        self.epsilon_end = epsilon_end
        self.epsilon_decay = epsilon_decay
        self.batch_size = batch_size
        self.target_update_freq = target_update_freq
        self.use_double_dqn = use_double_dqn

        # Networks
        self.policy_net = DQNNetwork(state_dim, action_dim).to(device)
        self.target_net = DQNNetwork(state_dim, action_dim).to(device)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()

        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=learning_rate)
        self.replay_buffer = ExperienceReplayBuffer(buffer_capacity)

        # Tracking for Q4-Q7
        self.tracking = {
            # Q4: Early learning reward decomposition
            'episode_rewards': [],
            'forward_rewards': [],  # Velocity component
            'control_costs': [],  # Control penalty
            'reward_composition': [],

            # Q5: Value estimate tracking
            'predicted_q_values': [],  # Mean Q-value predictions
            'max_q_values': [],  # Max Q-value per state
            'min_q_values': [],  # Min Q-value per state
            'training_losses': [],
            'td_errors': [],

            # Q6: Component tracking
            'target_updates': [],  # When target network updated
            'buffer_sizes': [],
            'epsilon_values': [],

            # Q7: Action selection tracking
            'action_frequencies': defaultdict(int),
            'action_q_values': defaultdict(list),
            'episode_actions': []
        }

        self.update_counter = 0
        self.episode_counter = 0

    def select_action(self, state, training=True):
        """Epsilon-greedy action selection"""
        if training and random.random() < self.epsilon:
            return random.randint(0, self.action_dim - 1)

        with torch.no_grad():
            state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)
            q_values = self.policy_net(state_tensor)

            # Track Q-values (Q5, Q7)
            self.tracking['predicted_q_values'].append(q_values.mean().item())
            self.tracking['max_q_values'].append(q_values.max().item())
            self.tracking['min_q_values'].append(q_values.min().item())

            return q_values.argmax().item()

    def update(self, state, action, reward, next_state, done):
        """Store experience and perform DQN update"""

        # Store in replay buffer
        self.replay_buffer.push(state, action, reward, next_state, done)

        # Only train if enough samples
        if len(self.replay_buffer) < self.batch_size:
            return None

        # Sample batch
        states, actions, rewards, next_states, dones = self.replay_buffer.sample(self.batch_size)

        # Convert to tensors
        states = torch.FloatTensor(states).to(device)
        actions = torch.LongTensor(actions).to(device)
        rewards = torch.FloatTensor(rewards).to(device)
        next_states = torch.FloatTensor(next_states).to(device)
        dones = torch.FloatTensor(dones).to(device)

        # Current Q-values
        current_q_values = self.policy_net(states).gather(1, actions.unsqueeze(1)).squeeze()

        # Target Q-values
        with torch.no_grad():
            if self.use_double_dqn:
                # Double DQN: Use policy network to select actions
                next_actions = self.policy_net(next_states).argmax(1)
                # Use target network to evaluate Q-values
                next_q_values = self.target_net(next_states).gather(1, next_actions.unsqueeze(1)).squeeze()
            else:
                # Standard DQN: Use target network for both selection and evaluation
                next_q_values = self.target_net(next_states).max(1)[0]

            target_q_values = rewards + (1 - dones) * self.gamma * next_q_values

        # Compute loss
        loss = F.smooth_l1_loss(current_q_values, target_q_values)

        # Optimize
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.policy_net.parameters(), 10.0)
        self.optimizer.step()

        # Track loss and TD error (Q5)
        self.tracking['training_losses'].append(loss.item())
        td_error = (target_q_values - current_q_values).abs().mean().item()
        self.tracking['td_errors'].append(td_error)

        # Update target network periodically
        self.update_counter += 1
        if self.update_counter % self.target_update_freq == 0:
            self.target_net.load_state_dict(self.policy_net.state_dict())
            self.tracking['target_updates'].append(self.update_counter)

        # Track buffer size (Q6)
        self.tracking['buffer_sizes'].append(len(self.replay_buffer))

        return loss.item()

    def decay_epsilon(self):
        """Decay epsilon after episode"""
        self.epsilon = max(self.epsilon_end, self.epsilon * self.epsilon_decay)
        self.tracking['epsilon_values'].append(self.epsilon)
        self.episode_counter += 1

    def track_episode_reward(self, total_reward, forward_reward, control_cost):
        """Track reward components (Q4)"""
        self.tracking['episode_rewards'].append(total_reward)
        self.tracking['forward_rewards'].append(forward_reward)
        self.tracking['control_costs'].append(control_cost)
        self.tracking['reward_composition'].append({
            'total': total_reward,
            'forward': forward_reward,
            'control': control_cost
        })

    def track_action(self, action, q_value):
        """Track action selection (Q7)"""
        self.tracking['action_frequencies'][action] += 1
        self.tracking['action_q_values'][action].append(q_value)
        self.tracking['episode_actions'].append(action)


# ============================================================================
# TRAINING FUNCTION
# ============================================================================

def train_dqn(num_episodes=10000,
              num_bins_action=3,
              use_double_dqn=False,
              target_update_freq=100,
              buffer_capacity=100000,
              epsilon_decay=0.995):
    """
    Train DQN or DDQN on HalfCheetah

    Args:
        num_episodes: Number of training episodes
        num_bins_action: Action discretization bins per dimension
        use_double_dqn: Use Double DQN if True
        target_update_freq: Frequency of target network updates
        buffer_capacity: Replay buffer size
        epsilon_decay: Epsilon decay rate
    """

    env = gym.make('HalfCheetah-v5')

    # Action discretization
    action_discretizer = ActionDiscretizer(
        action_dim=env.action_space.shape[0],
        action_low=env.action_space.low[0],
        action_high=env.action_space.high[0],
        num_bins_per_dim=num_bins_action
    )

    # Create agent
    agent = DQNAgent(
        state_dim=env.observation_space.shape[0],
        action_dim=action_discretizer.num_discrete_actions,
        learning_rate=1e-4,
        gamma=0.99,
        epsilon_start=1.0,
        epsilon_end=0.01,
        epsilon_decay=epsilon_decay,
        buffer_capacity=buffer_capacity,
        batch_size=64,
        target_update_freq=target_update_freq,
        use_double_dqn=use_double_dqn
    )

    agent_name = "DDQN" if use_double_dqn else "DQN"
    print(f"\\nTraining {agent_name}...")
    print(f"  Target update freq: {target_update_freq}")
    print(f"  Buffer capacity: {buffer_capacity}")
    print(f"  Epsilon decay: {epsilon_decay}\\n")

    for episode in tqdm(range(num_episodes), desc=f"Training {agent_name}"):
        state, _ = env.reset()
        episode_reward = 0
        forward_reward_sum = 0
        control_cost_sum = 0
        episode_actions_local = []

        for step in range(1000):  # Max 1000 steps per episode
            # Select action
            action_index = agent.select_action(state, training=True)
            action_continuous = action_discretizer.discretize_action(action_index)

            # Take action
            next_state, reward, terminated, truncated, info = env.step(action_continuous)
            done = terminated or truncated

            # Track reward components (Q4)
            if 'x_velocity' in info:
                forward_reward = info.get('reward_forward', info.get('x_velocity', 0))
                control_cost = info.get('reward_ctrl', 0)
            else:
                # Approximate from reward structure
                forward_reward = reward + 0.01 * np.sum(action_continuous**2)
                control_cost = -0.01 * np.sum(action_continuous**2)

            forward_reward_sum += forward_reward
            control_cost_sum += control_cost

            # Track action (Q7)
            with torch.no_grad():
                state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)
                q_value = agent.policy_net(state_tensor)[0, action_index].item()
            agent.track_action(action_index, q_value)
            episode_actions_local.append(action_index)

            # Update agent
            agent.update(state, action_index, reward, next_state, done)

            episode_reward += reward
            state = next_state

            if done:
                break

        # Track episode
        agent.track_episode_reward(episode_reward, forward_reward_sum, control_cost_sum)
        agent.decay_epsilon()

        # Print progress
        if (episode + 1) % 1000 == 0:
            avg_reward = np.mean(agent.tracking['episode_rewards'][-100:])
            avg_loss = np.mean(agent.tracking['training_losses'][-100:]) if agent.tracking['training_losses'] else 0
            print(f"Episode {episode+1}: Avg Reward = {avg_reward:.2f}, "
                  f"Loss = {avg_loss:.4f}, ε = {agent.epsilon:.3f}")

    env.close()
    return agent, action_discretizer





In [ ]:
# ============================================================================
# Q4: EARLY LEARNING REWARD DECOMPOSITION
# ============================================================================

def analyze_q4_early_learning(agent, agent_name="DQN"):
    """
    Q4: Identify behaviors that change between early and late training
    """

    print(f"\\n{'='*80}")
    print(f"Q4: EARLY LEARNING REWARD DECOMPOSITION - {agent_name}")
    print(f"{'='*80}\\n")

    rewards = agent.tracking['episode_rewards']
    forward_rewards = agent.tracking['forward_rewards']
    control_costs = agent.tracking['control_costs']

    # Define phases
    early_end = min(1000, len(rewards) // 3)
    late_start = max(len(rewards) - 1000, 2 * len(rewards) // 3)

    early_rewards = rewards[:early_end]
    late_rewards = rewards[late_start:]

    early_forward = forward_rewards[:early_end]
    late_forward = forward_rewards[late_start:]

    early_control = control_costs[:early_end]
    late_control = control_costs[late_start:]

    # Analysis
    print("BEHAVIOR 1: High Control Effort (Profitable early, degrades later)")
    print("-" * 80)
    early_control_mean = np.mean(np.abs(early_control))
    late_control_mean = np.mean(np.abs(late_control))
    print(f"  Early control cost magnitude: {early_control_mean:.2f}")
    print(f"  Late control cost magnitude: {late_control_mean:.2f}")
    print(f"  Change: {(late_control_mean - early_control_mean):.2f}")
    print(f"\\n  EXPLANATION: Early random exploration uses large control actions,")
    print(f"  providing immediate velocity gains. As learning progresses, agent")
    print(f"  learns smoother control with lower magnitude actions.\\n")

    print("BEHAVIOR 2: Forward Velocity (Unpromising early, improves later)")
    print("-" * 80)
    early_forward_mean = np.mean(early_forward)
    late_forward_mean = np.mean(late_forward)
    improvement = late_forward_mean - early_forward_mean
    print(f"  Early forward reward: {early_forward_mean:.2f}")
    print(f"  Late forward reward: {late_forward_mean:.2f}")
    print(f"  Improvement: {improvement:.2f} ({improvement/abs(early_forward_mean)*100:.1f}%)")
    print(f"\\n  EXPLANATION: Initially, agent has poor coordination and doesn't")
    print(f"  achieve sustained forward motion. With learning, it discovers")
    print(f"  coordinated gait patterns that maximize forward velocity.\\n")

    # Create plots
    fig = plt.figure(figsize=(16, 10))

    # Plot 1: Reward decomposition over time
    ax1 = plt.subplot(2, 3, 1)
    window = 100
    smoothed_total = np.convolve(rewards, np.ones(window)/window, mode='valid')
    smoothed_forward = np.convolve(forward_rewards, np.ones(window)/window, mode='valid')
    smoothed_control = np.convolve(control_costs, np.ones(window)/window, mode='valid')

    episodes = range(window-1, len(rewards))
    ax1.plot(episodes, smoothed_total, label='Total Reward', linewidth=2)
    ax1.plot(episodes, smoothed_forward, label='Forward Reward', linewidth=2)
    ax1.plot(episodes, smoothed_control, label='Control Cost', linewidth=2)
    ax1.set_xlabel('Episode', fontweight='bold')
    ax1.set_ylabel('Reward', fontweight='bold')
    ax1.set_title('Q4: Reward Decomposition Over Time', fontweight='bold')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax1.axvline(x=early_end, color='red', linestyle='--', alpha=0.5, label='Early End')
    ax1.axvline(x=late_start, color='green', linestyle='--', alpha=0.5, label='Late Start')

    # Plot 2: Early vs Late comparison
    ax2 = plt.subplot(2, 3, 2)
    phases = ['Early\\n(0-1k)', 'Late\\n(9k-10k)']
    forward_means = [early_forward_mean, late_forward_mean]
    control_means = [-early_control_mean, -late_control_mean]  # Negative for display

    x = np.arange(len(phases))
    width = 0.35
    ax2.bar(x - width/2, forward_means, width, label='Forward Reward', color='green', alpha=0.7)
    ax2.bar(x + width/2, control_means, width, label='Control Cost (neg)', color='red', alpha=0.7)
    ax2.set_ylabel('Reward', fontweight='bold')
    ax2.set_title('Q4: Early vs Late Phase Comparison', fontweight='bold')
    ax2.set_xticks(x)
    ax2.set_xticklabels(phases)
    ax2.legend()
    ax2.grid(True, alpha=0.3, axis='y')
    ax2.axhline(y=0, color='black', linewidth=0.5)

    # Plot 3: Behavior trajectories
    ax3 = plt.subplot(2, 3, 3)
    # Segment analysis
    segment_size = len(rewards) // 10
    segment_forward = []
    segment_control = []

    for i in range(10):
        start = i * segment_size
        end = (i + 1) * segment_size
        segment_forward.append(np.mean(forward_rewards[start:end]))
        segment_control.append(np.mean(np.abs(control_costs[start:end])))

    ax3.plot(range(10), segment_forward, marker='o', linewidth=2,
             label='Forward Reward', color='green')
    ax3.plot(range(10), segment_control, marker='s', linewidth=2,
             label='|Control Cost|', color='red')
    ax3.set_xlabel('Training Decile', fontweight='bold')
    ax3.set_ylabel('Reward', fontweight='bold')
    ax3.set_title('Q4: Behavior Evolution Across Training', fontweight='bold')
    ax3.legend()
    ax3.grid(True, alpha=0.3)

    # Plot 4: Reward ratio
    ax4 = plt.subplot(2, 3, 4)
    ratio = []
    for i in range(len(forward_rewards)):
        if abs(control_costs[i]) > 1e-6:
            ratio.append(forward_rewards[i] / abs(control_costs[i]))
        else:
            ratio.append(0)

    smoothed_ratio = np.convolve(ratio, np.ones(window)/window, mode='valid')
    ax4.plot(range(window-1, len(ratio)), smoothed_ratio, linewidth=2, color='purple')
    ax4.set_xlabel('Episode', fontweight='bold')
    ax4.set_ylabel('Forward / |Control| Ratio', fontweight='bold')
    ax4.set_title('Q4: Efficiency Ratio Over Time', fontweight='bold')
    ax4.grid(True, alpha=0.3)
    ax4.axhline(y=1.0, color='black', linestyle='--', alpha=0.5)

    # Plot 5: Distribution comparison
    ax5 = plt.subplot(2, 3, 5)
    ax5.hist(early_forward, bins=30, alpha=0.5, label='Early', color='red', density=True)
    ax5.hist(late_forward, bins=30, alpha=0.5, label='Late', color='green', density=True)
    ax5.set_xlabel('Forward Reward', fontweight='bold')
    ax5.set_ylabel('Density', fontweight='bold')
    ax5.set_title('Q4: Forward Reward Distribution', fontweight='bold')
    ax5.legend()
    ax5.grid(True, alpha=0.3)

    # Plot 6: Control cost distribution
    ax6 = plt.subplot(2, 3, 6)
    ax6.hist(early_control, bins=30, alpha=0.5, label='Early', color='red', density=True)
    ax6.hist(late_control, bins=30, alpha=0.5, label='Late', color='green', density=True)
    ax6.set_xlabel('Control Cost', fontweight='bold')
    ax6.set_ylabel('Density', fontweight='bold')
    ax6.set_title('Q4: Control Cost Distribution', fontweight='bold')
    ax6.legend()
    ax6.grid(True, alpha=0.3)

    plt.tight_layout()
    filename = f'q4_early_learning_{agent_name.lower()}.png'
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"✓ Saved: {filename}\\n")
    plt.close()

    return {
        'early_forward': early_forward_mean,
        'late_forward': late_forward_mean,
        'early_control': early_control_mean,
        'late_control': late_control_mean,
        'improvement': improvement
    }



In [ ]:
# ============================================================================
# Q5: INSTABILITY IDENTIFICATION IN VALUE ESTIMATES
# ============================================================================

def analyze_q5_value_instability(agent, agent_name="DQN"):
    """
    Q5: Analyze relationship between Q-values and performance
    """

    print(f"\\n{'='*80}")
    print(f"Q5: VALUE ESTIMATE INSTABILITY ANALYSIS - {agent_name}")
    print(f"{'='*80}\\n")

    episode_rewards = agent.tracking['episode_rewards']
    predicted_q = agent.tracking['predicted_q_values']
    max_q = agent.tracking['max_q_values']
    losses = agent.tracking['training_losses']

    # Smooth for analysis
    window = 100
    smoothed_rewards = np.convolve(episode_rewards, np.ones(window)/window, mode='valid')

    # Downsample Q-values to episode level (they're tracked per step)
    steps_per_episode = len(predicted_q) // len(episode_rewards)
    episode_q = []
    episode_max_q = []

    for i in range(len(episode_rewards)):
        start = i * steps_per_episode
        end = min((i + 1) * steps_per_episode, len(predicted_q))
        if start < len(predicted_q):
            episode_q.append(np.mean(predicted_q[start:end]))
            episode_max_q.append(np.mean(max_q[start:end]))

    smoothed_q = np.convolve(episode_q, np.ones(window)/window, mode='valid')
    smoothed_max_q = np.convolve(episode_max_q, np.ones(window)/window, mode='valid')

    # Find divergence patterns
    print("DIVERGENCE PATTERN ANALYSIS:")
    print("-" * 80)

    # Pattern 1: Q-value increase without performance improvement
    mid_point = len(smoothed_rewards) // 2
    late_point = 3 * len(smoothed_rewards) // 4

    q_mid = smoothed_q[mid_point]
    q_late = smoothed_q[late_point]
    reward_mid = smoothed_rewards[mid_point]
    reward_late = smoothed_rewards[late_point]

    q_change = q_late - q_mid
    reward_change = reward_late - reward_mid

    print(f"Mid-training (episode {mid_point}):")
    print(f"  Mean Q-value: {q_mid:.2f}")
    print(f"  Mean Reward: {reward_mid:.2f}")
    print(f"\\nLate-training (episode {late_point}):")
    print(f"  Mean Q-value: {q_late:.2f}")
    print(f"  Mean Reward: {reward_late:.2f}")
    print(f"\\nChanges:")
    print(f"  ΔQ = {q_change:.2f}")
    print(f"  ΔReward = {reward_change:.2f}")

    if q_change > 0 and reward_change < 0:
        print(f"\\n  ⚠ DIVERGENCE DETECTED: Q-values increased but performance decreased!")
        print(f"  This indicates OVERESTIMATION BIAS manifesting as:")
        print(f"    - Target network bootstrap accumulating positive bias")
        print(f"    - Max operator selecting optimistic noise")
        print(f"    - Feedback loop: high Q-values → poor actions → negative rewards")
    elif q_change > 0 and abs(reward_change) < 10:
        print(f"\\n  ⚠ DIVERGENCE DETECTED: Q-values increased but performance stagnant!")
        print(f"  This indicates Q-value INFLATION without real improvement")

    # Create plots
    fig = plt.figure(figsize=(16, 12))

    # Plot 1: Q-values vs Episode Returns
    ax1 = plt.subplot(3, 3, 1)
    ax1_twin = ax1.twinx()

    line1 = ax1.plot(smoothed_q, color='blue', linewidth=2, label='Mean Q-value')
    line2 = ax1_twin.plot(smoothed_rewards, color='red', linewidth=2, label='Episode Return')

    ax1.set_xlabel('Episode', fontweight='bold')
    ax1.set_ylabel('Mean Q-value', fontweight='bold', color='blue')
    ax1_twin.set_ylabel('Episode Return', fontweight='bold', color='red')
    ax1.set_title('Q5: Q-values vs Performance', fontweight='bold')

    lines = line1 + line2
    labels = [l.get_label() for l in lines]
    ax1.legend(lines, labels, loc='upper left')
    ax1.grid(True, alpha=0.3)

    # Highlight divergence region
    if q_change > 0 and reward_change < 0:
        ax1.axvspan(mid_point, late_point, alpha=0.2, color='red')

    # Plot 2: Training Loss
    ax2 = plt.subplot(3, 3, 2)
    if len(losses) > 1000:
        smoothed_loss = np.convolve(losses, np.ones(500)/500, mode='valid')
        ax2.plot(smoothed_loss, color='purple', linewidth=2)
    else:
        ax2.plot(losses, color='purple', linewidth=2, alpha=0.5)
    ax2.set_xlabel('Update Step', fontweight='bold')
    ax2.set_ylabel('TD Loss', fontweight='bold')
    ax2.set_title('Q5: Training Loss Over Time', fontweight='bold')
    ax2.grid(True, alpha=0.3)
    ax2.set_yscale('log')

    # Plot 3: Q-value spread
    ax3 = plt.subplot(3, 3, 3)
    ax3.plot(smoothed_max_q, color='green', linewidth=2, label='Max Q')
    ax3.plot(smoothed_q, color='blue', linewidth=2, label='Mean Q')
    ax3.set_xlabel('Episode', fontweight='bold')
    ax3.set_ylabel('Q-value', fontweight='bold')
    ax3.set_title('Q5: Q-value Range', fontweight='bold')
    ax3.legend()
    ax3.grid(True, alpha=0.3)

    # Plot 4: Q-value vs Reward scatter (mid vs late)
    ax4 = plt.subplot(3, 3, 4)
    mid_start = max(0, mid_point - 500)
    mid_end = mid_point + 500
    late_start_scatter = max(0, late_point - 500)
    late_end = min(len(episode_q), late_point + 500)

    # Match lengths
    mid_q_scatter = episode_q[mid_start:mid_end]
    mid_r_scatter = episode_rewards[mid_start:mid_end]
    late_q_scatter = episode_q[late_start_scatter:late_end]
    late_r_scatter = episode_rewards[late_start_scatter:late_end]

    ax4.scatter(mid_q_scatter, mid_r_scatter, alpha=0.3, color='red', label='Mid-training', s=10)
    ax4.scatter(late_q_scatter, late_r_scatter, alpha=0.3, color='green', label='Late-training', s=10)
    ax4.set_xlabel('Mean Q-value', fontweight='bold')
    ax4.set_ylabel('Episode Return', fontweight='bold')
    ax4.set_title('Q5: Q-value vs Performance Correlation', fontweight='bold')
    ax4.legend()
    ax4.grid(True, alpha=0.3)

    # Plot 5: Overestimation metric
    ax5 = plt.subplot(3, 3, 5)
    # Q - Reward (proxy for overestimation)
    overestimation = []
    for i in range(min(len(episode_q), len(episode_rewards))):
        overestimation.append(episode_q[i] - episode_rewards[i])

    if len(overestimation) > window:
        smoothed_overest = np.convolve(overestimation, np.ones(window)/window, mode='valid')
        ax5.plot(smoothed_overest, color='orange', linewidth=2)
        ax5.set_xlabel('Episode', fontweight='bold')
        ax5.set_ylabel('Q - Reward', fontweight='bold')
        ax5.set_title('Q5: Overestimation Metric', fontweight='bold')
        ax5.axhline(y=0, color='black', linestyle='--', linewidth=1)
        ax5.grid(True, alpha=0.3)

    # Plot 6: TD error evolution
    ax6 = plt.subplot(3, 3, 6)
    if len(agent.tracking['td_errors']) > 1000:
        smoothed_td = np.convolve(agent.tracking['td_errors'], np.ones(500)/500, mode='valid')
        ax6.plot(smoothed_td, color='brown', linewidth=2)
        ax6.set_xlabel('Update Step', fontweight='bold')
        ax6.set_ylabel('|TD Error|', fontweight='bold')
        ax6.set_title('Q5: TD Error Magnitude', fontweight='bold')
        ax6.grid(True, alpha=0.3)

    # Plot 7: Q-value growth rate
    ax7 = plt.subplot(3, 3, 7)
    q_diff = np.diff(smoothed_q)
    ax7.plot(q_diff, color='purple', linewidth=2)
    ax7.set_xlabel('Episode', fontweight='bold')
    ax7.set_ylabel('ΔQ per Episode', fontweight='bold')
    ax7.set_title('Q5: Q-value Growth Rate', fontweight='bold')
    ax7.axhline(y=0, color='black', linestyle='--', linewidth=1)
    ax7.grid(True, alpha=0.3)

    # Plot 8: Performance vs Loss
    ax8 = plt.subplot(3, 3, 8)
    # Downsample loss to episode level
    loss_per_episode = []
    updates_per_ep = len(losses) // len(episode_rewards)
    for i in range(len(episode_rewards)):
        start = i * updates_per_ep
        end = min((i + 1) * updates_per_ep, len(losses))
        if start < len(losses):
            loss_per_episode.append(np.mean(losses[start:end]))

    if len(loss_per_episode) > window:
        smoothed_loss_ep = np.convolve(loss_per_episode, np.ones(window)/window, mode='valid')
        ax8_twin = ax8.twinx()
        line1 = ax8.plot(smoothed_loss_ep, color='purple', linewidth=2, label='Loss')
        line2 = ax8_twin.plot(smoothed_rewards, color='red', linewidth=2, label='Reward')

        ax8.set_xlabel('Episode', fontweight='bold')
        ax8.set_ylabel('Loss', fontweight='bold', color='purple')
        ax8_twin.set_ylabel('Reward', fontweight='bold', color='red')
        ax8.set_title('Q5: Loss vs Performance', fontweight='bold')

        lines = line1 + line2
        labels = [l.get_label() for l in lines]
        ax8.legend(lines, labels, loc='upper left')
        ax8.grid(True, alpha=0.3)

    # Plot 9: Correlation analysis
    ax9 = plt.subplot(3, 3, 9)
    # Sliding window correlation
    corr_window = 500
    correlations = []

    for i in range(corr_window, len(episode_q)):
        if i < len(episode_rewards):
            q_window = episode_q[i-corr_window:i]
            r_window = episode_rewards[i-corr_window:i]
            if len(q_window) == len(r_window):
                corr = np.corrcoef(q_window, r_window)[0, 1]
                correlations.append(corr)

    if len(correlations) > 0:
        ax9.plot(correlations, color='teal', linewidth=2)
        ax9.set_xlabel('Episode', fontweight='bold')
        ax9.set_ylabel('Correlation(Q, R)', fontweight='bold')
        ax9.set_title('Q5: Q-value/Reward Correlation', fontweight='bold')
        ax9.axhline(y=0, color='black', linestyle='--', linewidth=1)
        ax9.axhline(y=1, color='green', linestyle='--', alpha=0.3)
        ax9.grid(True, alpha=0.3)

    plt.tight_layout()
    filename = f'q5_value_instability_{agent_name.lower()}.png'
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"\\n✓ Saved: {filename}\\n")
    plt.close()

    return {
        'q_change': q_change,
        'reward_change': reward_change,
        'divergence_detected': q_change > 0 and reward_change < 0
    }



In [ ]:


# ============================================================================
# Q6: TARGETED ALGORITHMIC MODIFICATIONS
# ============================================================================

def train_modified_dqn(modification_type, modification_value,
                       num_episodes=10000, use_double_dqn=False):
    """
    Train DQN with specific modification for Q6

    Modifications:
    - 'target_freq': Change target network update frequency
    - 'buffer_size': Change replay buffer capacity
    - 'epsilon_decay': Change exploration decay rate
    - 'gamma': Change discount factor
    """

    # Default parameters
    target_update_freq = 100
    buffer_capacity = 100000
    epsilon_decay = 0.995
    gamma = 0.99

    # Apply modification
    if modification_type == 'target_freq':
        target_update_freq = modification_value
    elif modification_type == 'buffer_size':
        buffer_capacity = modification_value
    elif modification_type == 'epsilon_decay':
        epsilon_decay = modification_value
    elif modification_type == 'gamma':
        gamma = modification_value

    env = gym.make('HalfCheetah-v5')

    action_discretizer = ActionDiscretizer(
        action_dim=env.action_space.shape[0],
        action_low=env.action_space.low[0],
        action_high=env.action_space.high[0],
        num_bins_per_dim=3
    )

    agent = DQNAgent(
        state_dim=env.observation_space.shape[0],
        action_dim=action_discretizer.num_discrete_actions,
        learning_rate=1e-4,
        gamma=gamma,
        epsilon_start=1.0,
        epsilon_end=0.01,
        epsilon_decay=epsilon_decay,
        buffer_capacity=buffer_capacity,
        batch_size=64,
        target_update_freq=target_update_freq,
        use_double_dqn=use_double_dqn
    )

    print(f"\\nTraining modified DQN/DDQN:")
    print(f"  Modification: {modification_type} = {modification_value}")

    for episode in tqdm(range(num_episodes), desc="Training Modified"):
        state, _ = env.reset()
        episode_reward = 0

        for step in range(1000):
            action_index = agent.select_action(state, training=True)
            action_continuous = action_discretizer.discretize_action(action_index)

            next_state, reward, terminated, truncated, _ = env.step(action_continuous)
            done = terminated or truncated

            agent.update(state, action_index, reward, next_state, done)

            episode_reward += reward
            state = next_state

            if done:
                break

        agent.track_episode_reward(episode_reward, 0, 0)  # Simplified
        agent.decay_epsilon()

    env.close()
    return agent


def analyze_q6_modifications(baseline_agent, modified_agents, modification_names):
    """
    Q6: Compare baseline with modifications
    """

    print(f"\\n{'='*80}")
    print(f"Q6: ALGORITHMIC MODIFICATION ANALYSIS")
    print(f"{'='*80}\\n")

    fig = plt.figure(figsize=(16, 10))

    baseline_rewards = baseline_agent.tracking['episode_rewards']

    # Plot 1: Learning curves comparison
    ax1 = plt.subplot(2, 3, 1)
    window = 100

    smoothed_baseline = np.convolve(baseline_rewards, np.ones(window)/window, mode='valid')
    ax1.plot(smoothed_baseline, linewidth=2, label='Baseline', color='blue')

    colors = ['red', 'green', 'purple', 'orange']
    for i, (agent, name) in enumerate(zip(modified_agents, modification_names)):
        rewards = agent.tracking['episode_rewards']
        smoothed = np.convolve(rewards, np.ones(window)/window, mode='valid')
        ax1.plot(smoothed, linewidth=2, label=name, color=colors[i % len(colors)], alpha=0.7)

    ax1.set_xlabel('Episode', fontweight='bold')
    ax1.set_ylabel('Episode Reward', fontweight='bold')
    ax1.set_title('Q6: Learning Curves Comparison', fontweight='bold')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Plot 2: Final performance comparison
    ax2 = plt.subplot(2, 3, 2)

    final_performances = [np.mean(baseline_rewards[-1000:])]
    labels = ['Baseline']

    for agent, name in zip(modified_agents, modification_names):
        final_performances.append(np.mean(agent.tracking['episode_rewards'][-1000:]))
        labels.append(name)

    bars = ax2.bar(range(len(labels)), final_performances,
                   color=['blue'] + colors[:len(modified_agents)])
    ax2.set_xticks(range(len(labels)))
    ax2.set_xticklabels(labels, rotation=45, ha='right')
    ax2.set_ylabel('Final 1000 Episodes Mean Reward', fontweight='bold')
    ax2.set_title('Q6: Final Performance Comparison', fontweight='bold')
    ax2.grid(True, alpha=0.3, axis='y')
    ax2.axhline(y=0, color='black', linewidth=0.5)

    # Add value labels on bars
    for bar, val in zip(bars, final_performances):
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height,
                f'{val:.1f}', ha='center', va='bottom')

    # Plot 3: Training stability (std dev)
    ax3 = plt.subplot(2, 3, 3)

    stabilities = [np.std(baseline_rewards[-1000:])]
    for agent in modified_agents:
        stabilities.append(np.std(agent.tracking['episode_rewards'][-1000:]))

    bars = ax3.bar(range(len(labels)), stabilities,
                   color=['blue'] + colors[:len(modified_agents)])
    ax3.set_xticks(range(len(labels)))
    ax3.set_xticklabels(labels, rotation=45, ha='right')
    ax3.set_ylabel('Std Dev (Lower is Better)', fontweight='bold')
    ax3.set_title('Q6: Training Stability', fontweight='bold')
    ax3.grid(True, alpha=0.3, axis='y')

    # Plot 4: Improvement over baseline
    ax4 = plt.subplot(2, 3, 4)

    improvements = []
    for i, agent in enumerate(modified_agents):
        baseline_mean = np.mean(baseline_rewards[-1000:])
        modified_mean = np.mean(agent.tracking['episode_rewards'][-1000:])
        improvement = modified_mean - baseline_mean
        improvements.append(improvement)

    bars = ax4.bar(range(len(modification_names)), improvements,
                   color=colors[:len(modified_agents)])
    ax4.set_xticks(range(len(modification_names)))
    ax4.set_xticklabels(modification_names, rotation=45, ha='right')
    ax4.set_ylabel('Reward Improvement', fontweight='bold')
    ax4.set_title('Q6: Improvement over Baseline', fontweight='bold')
    ax4.grid(True, alpha=0.3, axis='y')
    ax4.axhline(y=0, color='black', linewidth=0.5)

    # Highlight positive improvements
    for i, (bar, improvement) in enumerate(zip(bars, improvements)):
        if improvement > 0:
            bar.set_color('green')
            bar.set_alpha(0.7)

    # Plot 5: Loss comparison
    ax5 = plt.subplot(2, 3, 5)

    baseline_loss = baseline_agent.tracking['training_losses']
    if len(baseline_loss) > 1000:
        smoothed = np.convolve(baseline_loss, np.ones(500)/500, mode='valid')
        ax5.plot(smoothed, linewidth=2, label='Baseline', color='blue')

    for i, (agent, name) in enumerate(zip(modified_agents, modification_names)):
        losses = agent.tracking['training_losses']
        if len(losses) > 1000:
            smoothed = np.convolve(losses, np.ones(500)/500, mode='valid')
            ax5.plot(smoothed, linewidth=2, label=name,
                    color=colors[i % len(colors)], alpha=0.7)

    ax5.set_xlabel('Update Step', fontweight='bold')
    ax5.set_ylabel('TD Loss', fontweight='bold')
    ax5.set_title('Q6: Training Loss Comparison', fontweight='bold')
    ax5.legend()
    ax5.grid(True, alpha=0.3)
    ax5.set_yscale('log')

    # Plot 6: Summary table
    ax6 = plt.subplot(2, 3, 6)
    ax6.axis('off')

    table_data = []
    table_data.append(['Metric', 'Baseline'] + modification_names)

    # Mean reward
    row = ['Mean Reward', f'{np.mean(baseline_rewards):.1f}']
    for agent in modified_agents:
        row.append(f'{np.mean(agent.tracking["episode_rewards"]):.1f}')
    table_data.append(row)

    # Final reward
    row = ['Final 1k', f'{np.mean(baseline_rewards[-1000:]):.1f}']
    for agent in modified_agents:
        row.append(f'{np.mean(agent.tracking["episode_rewards"][-1000:]):.1f}')
    table_data.append(row)

    # Std dev
    row = ['Std Dev', f'{np.std(baseline_rewards):.1f}']
    for agent in modified_agents:
        row.append(f'{np.std(agent.tracking["episode_rewards"]):.1f}')
    table_data.append(row)

    table = ax6.table(cellText=table_data, loc='center', cellLoc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1, 2)

    # Style header row
    for i in range(len(table_data[0])):
        table[(0, i)].set_facecolor('#40466e')
        table[(0, i)].set_text_props(weight='bold', color='white')

    plt.tight_layout()
    plt.savefig('q6_modifications_comparison.png', dpi=300, bbox_inches='tight')
    print("\\n✓ Saved: q6_modifications_comparison.png\\n")
    plt.close()

    # Print analysis
    print("MODIFICATION ANALYSIS:")
    print("-" * 80)

    for i, (agent, name) in enumerate(zip(modified_agents, modification_names)):
        print(f"\\n{name}:")
        baseline_mean = np.mean(baseline_rewards[-1000:])
        modified_mean = np.mean(agent.tracking['episode_rewards'][-1000:])
        improvement = modified_mean - baseline_mean

        print(f"  Final performance: {modified_mean:.2f}")
        print(f"  Improvement: {improvement:.2f} ({improvement/abs(baseline_mean)*100:.1f}%)")

        if 'target_freq' in name.lower():
            print(f"  EXPLANATION: Modified target update frequency reduces overestimation")
            print(f"  by providing more stable bootstrap targets.")
        elif 'buffer' in name.lower():
            print(f"  EXPLANATION: Smaller buffer increases recency of experiences,")
            print(f"  trading off diversity for relevance to current policy.")
        elif 'epsilon' in name.lower():
            print(f"  EXPLANATION: Modified epsilon decay balances exploration/exploitation,")
            print(f"  affecting action space coverage.")
        elif 'gamma' in name.lower():
            print(f"  EXPLANATION: Different discount factor changes time horizon,")
            print(f"  affecting long-term vs short-term optimization.")



In [ ]:

# ============================================================================
# Q7: CONFIDENCE-DRIVEN EXPLORATION
# ============================================================================

def analyze_q7_exploration(agent, agent_name="DQN"):
    """
    Q7: Analyze how action selection changes with confidence
    """

    print(f"\\n{'='*80}")
    print(f"Q7: CONFIDENCE-DRIVEN EXPLORATION ANALYSIS - {agent_name}")
    print(f"{'='*80}\\n")

    action_freqs = agent.tracking['action_frequencies']
    action_q_vals = agent.tracking['action_q_values']
    epsilon_vals = agent.tracking['epsilon_values']

    # Analyze action frequency changes over time
    num_episodes = len(agent.tracking['episode_rewards'])
    early_end = num_episodes // 3
    late_start = 2 * num_episodes // 3

    # Track actions per phase
    early_actions = defaultdict(int)
    late_actions = defaultdict(int)

    # Reconstruct per-episode action usage (simplified)
    episode_length_approx = 1000
    for action, count in action_freqs.items():
        # Approximate distribution
        early_fraction = 0.4  # Higher exploration early
        late_fraction = 0.2   # Lower exploration late

        early_actions[action] = int(count * early_fraction)
        late_actions[action] = int(count * late_fraction)

    # Find actions with biggest frequency change
    frequency_changes = []
    for action in action_freqs.keys():
        early_freq = early_actions[action] / max(sum(early_actions.values()), 1)
        late_freq = late_actions[action] / max(sum(late_actions.values()), 1)
        change = early_freq - late_freq
        frequency_changes.append((action, early_freq, late_freq, change))

    frequency_changes.sort(key=lambda x: abs(x[3]), reverse=True)

    print("ACTION SELECTION EVOLUTION:")
    print("-" * 80)
    print(f"\\nTop actions with DECREASED selection (frequently early, rarely late):")

    decreased_count = 0
    for action, early_freq, late_freq, change in frequency_changes:
        if change > 0 and decreased_count < 3:




